## Model Development Notebook — TF-IDF, English-Only

**Why this version differs from the original:** the multilingual sentence-embedding
approach (`sentence-transformers` + `torch`) could not fit within free-tier hosting
memory limits (Render: 512MB, Hugging Face free tier: GPU-only). This version switches
to **TF-IDF** features, which removes the `torch`/deep-learning dependency entirely,
at the cost of dropping German-language support and cross-lingual understanding.

**Scope change:** English-only. German rows are filtered out early. This is a
documented, deliberate trade-off made to fit a real infrastructure constraint, not
an oversight.

**Phases:**
1. Load raw dataset, clean (missing values, duplicates, filter to official 10 queues)
2. Filter to English only
3. Combine `subject` + `body` into `text`
4. Generate sentiment labels (local pretrained model, since ground truth doesn't exist)
5. Stratified train/val/test split
6. TF-IDF vectorization
7. Train + evaluate intent, sentiment, and priority classifiers (untuned LR baseline, then tuned comparison)
8. Save the vectorizer and all three models


In [ ]:
!pip install transformers torch --q

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

from datasets import load_dataset
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

from transformers import pipeline

### Phase 1 — Load raw dataset

In [ ]:
ds = load_dataset("Tobi-Bueck/customer-support-tickets")
df = ds["train"].to_pandas()
print(df.shape)
df.head()

### Phase 1 continued — Cleaning

Same cleaning steps as the original pipeline:
- Drop rows with missing `body` (essential field, only 2 rows affected)
- Fill missing `subject` with empty string (8.5% missing, don't lose the row over it)
- Drop exact duplicate rows
- Filter to the 10 official NorthStack queues (the raw dataset contains many
  off-topic categories like "Pets & Animals" that don't belong to a SaaS company)

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=['body'])
df['subject'] = df['subject'].fillna('')
print(df.shape)

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

In [ ]:
official_queues = [
    'Billing and Payments', 'Customer Service', 'General Inquiry',
    'Human Resources', 'IT Support', 'Product Support',
    'Returns and Exchanges', 'Sales and Pre-Sales',
    'Service Outages and Maintenance', 'Technical Support'
]

df2 = df[df['queue'].isin(official_queues)].copy()
print(df2.shape)
df2['queue'].value_counts()

### Phase 2 — Filter to English only

This is the key scope change from the original approach. German rows are dropped
entirely — TF-IDF has no mechanism to relate English and German vocabulary to each
other, so a single mixed-vocabulary model would perform poorly on both. Restricting
to one language keeps the model coherent, at the cost of German support.

In [ ]:
print("Before filtering:")
print(df2['language'].value_counts())

df2 = df2[df2['language'] == 'en'].reset_index(drop=True)

print("\nAfter filtering to English only:")
print(df2.shape)
print(df2['queue'].value_counts())

### Phase 3 — Combine subject + body into text

In [ ]:
df2['text'] = (df2['subject'] + ' ' + df2['body']).str.strip()
df2[['subject', 'body', 'text']].head()

### Phase 4 — Generate sentiment labels

Sentiment doesn't exist in the raw data, so it's generated using a local pretrained
model (satisfies the "no external API calls" constraint) — same approach as the
original pipeline, just run on the smaller English-only subset now, which is faster.

In [ ]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

In [ ]:
def map_stars_to_sentiment(label):
    star = int(label[0])
    if star <= 2:
        return "Negative"
    elif star == 3:
        return "Neutral"
    else:
        return "Positive"

texts = df2['text'].str.slice(0, 512).tolist()
results = sentiment_model(texts, batch_size=32, truncation=True)

df2['sentiment_raw'] = [r['label'] for r in results]
df2['sentiment'] = df2['sentiment_raw'].apply(map_stars_to_sentiment)

df2['sentiment'].value_counts()

In [ ]:
# Sanity check against priority - Critical/high priority should skew more negative
pd.crosstab(df2['priority'], df2['sentiment'], normalize='index')

In [ ]:
# Save this cleaned + labeled dataframe so it doesn't need regenerating if the
# notebook session restarts
df2.to_csv('data_queue_sentiment_en.csv', index=False)
print("Saved.")

### Phase 5 — Stratified train/val/test split

In [ ]:
train_df, temp_df = train_test_split(
    df2, test_size=0.30, stratify=df2['queue'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['queue'], random_state=42
)

print(train_df.shape, val_df.shape, test_df.shape)

### Phase 6 — TF-IDF vectorization

Fit on the training set only, then transform val/test — fitting on the full dataset
would leak word-frequency information from val/test into training. This is a new
leakage risk that didn't apply with the embedding approach (each row was encoded
independently there), worth remembering as a real difference between the two methods.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english'
)

X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

print(X_train.shape, X_val.shape, X_test.shape)

### Phase 7 — Modeling

Baseline is an **untuned Logistic Regression** (not a dummy classifier) — same
discipline as the original embedding-based pipeline. Then a tuned multi-model
comparison (LR, SVM, Random Forest, XGBoost) for each of the three targets.

In [ ]:
model_configs = {
    'Logistic Regression': {
        'estimator': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'param_dist': {'C': [0.01, 0.1, 1, 10, 100]},
        'uses_encoded_labels': False
    },
    'SVM': {
        'estimator': SVC(class_weight='balanced', random_state=42),
        'param_dist': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']},
        'uses_encoded_labels': False
    },
    'Random Forest': {
        'estimator': RandomForestClassifier(class_weight='balanced', random_state=42),
        'param_dist': {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, None]},
        'uses_encoded_labels': False
    },
    'XGBoost': {
        'estimator': xgb.XGBClassifier(random_state=42, eval_metric='mlogloss'),
        'param_dist': {'n_estimators': [100, 200], 'max_depth': [3, 6, 9], 'learning_rate': [0.01, 0.1, 0.3]},
        'uses_encoded_labels': True
    }
}

def run_baseline(y_train, y_val, label_name):
    baseline = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    baseline.fit(X_train, y_train)
    print(f"=== Untuned Logistic Regression Baseline ({label_name}) ===")
    print(classification_report(y_val, baseline.predict(X_val)))
    return baseline

def run_comparison(y_train, y_val, label_name):
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)

    results = []
    fitted = {}

    for name, config in model_configs.items():
        print(f"--- Training {name} ({label_name}) ---")
        y_tr = y_train_enc if config['uses_encoded_labels'] else y_train
        y_va = y_val_enc if config['uses_encoded_labels'] else y_val

        search = RandomizedSearchCV(
            config['estimator'], config['param_dist'],
            n_iter=5, cv=3, scoring='f1_macro', random_state=42, n_jobs=-1
        )
        search.fit(X_train, y_tr)

        y_val_pred = search.predict(X_val)
        val_f1_macro = f1_score(y_va, y_val_pred, average='macro')

        results.append({'model': name, 'best_params': search.best_params_, 'val_f1_macro': val_f1_macro})
        fitted[name] = search.best_estimator_
        print(f"{name} — val macro F1: {val_f1_macro:.4f}")

    return pd.DataFrame(results).sort_values('val_f1_macro', ascending=False), fitted

#### 7a — Intent (`queue`)

In [ ]:
y_train_intent = train_df['queue']
y_val_intent = val_df['queue']

baseline_intent = run_baseline(y_train_intent, y_val_intent, "intent")

In [ ]:
results_intent_df, fitted_models_intent = run_comparison(y_train_intent, y_val_intent, "intent")
results_intent_df

In [ ]:
best_intent_name = results_intent_df.iloc[0]['model']
best_intent_model = fitted_models_intent[best_intent_name]
print(f"Best model: {best_intent_name}")
print(classification_report(y_val_intent, best_intent_model.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val_intent, best_intent_model.predict(X_val), labels=sorted(y_val_intent.unique()))
import seaborn as sns
import matplotlib.pyplot as plt

labels = sorted(y_val_intent.unique())
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_intent_name} (Intent, English-only, TF-IDF)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#### 7b — Sentiment

In [ ]:
y_train_sent = train_df['sentiment']
y_val_sent = val_df['sentiment']

baseline_sent = run_baseline(y_train_sent, y_val_sent, "sentiment")

In [ ]:
results_sent_df, fitted_models_sent = run_comparison(y_train_sent, y_val_sent, "sentiment")
results_sent_df

In [ ]:
best_sent_name = results_sent_df.iloc[0]['model']
best_sent_model = fitted_models_sent[best_sent_name]
print(f"Best model: {best_sent_name}")
print(classification_report(y_val_sent, best_sent_model.predict(X_val)))

#### 7c — Priority

In [ ]:
y_train_pri = train_df['priority']
y_val_pri = val_df['priority']

baseline_pri = run_baseline(y_train_pri, y_val_pri, "priority")

In [ ]:
results_pri_df, fitted_models_pri = run_comparison(y_train_pri, y_val_pri, "priority")
results_pri_df

In [ ]:
best_pri_name = results_pri_df.iloc[0]['model']
best_pri_model = fitted_models_pri[best_pri_name]
print(f"Best model: {best_pri_name}")
print(classification_report(y_val_pri, best_pri_model.predict(X_val)))

### Phase 8 — Save the vectorizer and all three models

**Important:** the `TfidfVectorizer` itself must be saved and loaded alongside the
models, the same way the embedder had to be before — this vectorizer was fit
specifically on the training vocabulary, and a different/unfitted vectorizer would
produce an incompatible feature space.

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(vectorizer, 'models/tfidf_vectorizer.joblib')
joblib.dump(best_intent_model, 'models/intent_classifier.joblib')
joblib.dump(best_sent_model, 'models/sentiment_classifier.joblib')
joblib.dump(best_pri_model, 'models/priority_classifier.joblib')

print("Saved:")
print(f"  Vectorizer")
print(f"  Intent model: {best_intent_name}")
print(f"  Sentiment model: {best_sent_name}")
print(f"  Priority model: {best_pri_name}")

In [ ]:
# Sanity check: reload and confirm identical predictions
loaded_vectorizer = joblib.load('models/tfidf_vectorizer.joblib')
loaded_intent = joblib.load('models/intent_classifier.joblib')

sample_text = test_df['text'].iloc[:5]
X_sample = loaded_vectorizer.transform(sample_text)

original_pred = best_intent_model.predict(X_test[:5])
loaded_pred = loaded_intent.predict(X_sample)

print("Match:", (original_pred == loaded_pred).all())

### Final test-set evaluation (held out, seen only now)

Report these numbers — not the validation numbers above — as the final, honest
performance figures in your PRD/report.

In [ ]:
y_test_intent = test_df['queue']
y_test_sent = test_df['sentiment']
y_test_pri = test_df['priority']

print("=== FINAL TEST SET RESULTS ===\n")

print(f"--- Intent ({best_intent_name}) ---")
print(classification_report(y_test_intent, best_intent_model.predict(X_test)))

print(f"--- Sentiment ({best_sent_name}) ---")
print(classification_report(y_test_sent, best_sent_model.predict(X_test)))

print(f"--- Priority ({best_pri_name}) ---")
print(classification_report(y_test_pri, best_pri_model.predict(X_test)))

### Note on what changed vs. the original multilingual/embedding approach

- **Dropped**: German-language support, semantic/cross-lingual understanding, the
  `sentence-transformers`/`torch` dependency at inference time
- **Gained**: a dramatically smaller memory footprint (no deep learning library
  needed for serving), making the models deployable on free-tier hosting (Render, 512MB RAM)
- **Trade-off to report honestly**: TF-IDF treats words as isolated symbols with no
  semantic understanding — synonyms and paraphrases that the embedding model would
  have recognized as similar now look unrelated. Compare the test-set F1 scores here
  against the original embedding-based results to quantify the actual accuracy cost
  of this decision, and report both numbers side by side in the PRD.
- **Note**: the sentiment-generation step still uses a local pretrained transformer
  model (`nlptown/bert-base-multilingual-uncased-sentiment`) — this only runs here,
  in the notebook, at training/labeling time. It is NOT part of the deployed
  `classify.py`, so it does not affect the deployed app's memory footprint.
